In [1]:
import os
import json
from typing import Iterable
from pyi18next.i18next import I18next
from pyi18next.backends.fs import Backend
from pyi18next.utility import get_plural_func
from services.dense_vector_engine import DenseVectorEngine
from pprint import pprint
from core.settings import get_settings
from controllers.load_models import resolve_models
from controllers.transformer import Transformer
import re
from controllers.load_models import (
	get_sberts, 
	get_berts, 
	get_siamese_lstms,
	LazyLoader
)


In [2]:
settings = get_settings()


In [3]:
def traverse_namespaces(base_path: str, languages: Iterable[str]):
	namespaces = set()

	for lng in languages:
		lng_path = os.path.join(base_path, lng)

		if os.path.isdir(lng_path):
			for root, _, files in os.walk(lng_path):
				for file in files:
					if file.endswith(".json"):
						full_path = os.path.join(root, file)
						
						rel_path = os.path.relpath(full_path, lng_path)

						namespace = os.path.splitext(rel_path)[0]
						namespace = namespace.replace(os.sep, "/")

						namespaces.add(namespace)

	return list(namespaces)

namespaces = traverse_namespaces("localization", settings.languages)
print(namespaces)


['scene4/scene4Frontyard', 'scene6/routeA/scene6PortalRouteA', 'menus/creditsScene', 'names', 'scene7/scene7Bedroom', 'scene1/scene1Bedroom1', 'scene1/scene1Classroom', 'generalDialogs', 'transitions', 'scene3/scene3Bedroom', 'scene6/scene6Livingroom', 'scene6/routeB/scene6LunchRouteB', 'scene6/routeA/scene6BedroomRouteA2', 'scene5/scene5Bedroom', 'scene4/scene4Garage', 'scene6/routeB/scene6BedroomRouteB', 'scene4/scene4Bedroom', 'scene6/routeB/scene6PoliceStationRouteB', 'computer/socialMediaScreen', 'menus/loginScene', 'dialogManager', 'scene5/scene5Livingroom', 'scene6/routeB/scene6EndingRouteB', 'scene1/scene1Break', 'scene3/scene3Break', 'scene6/routeA/scene6BedroomRouteA1', 'scene1/scene1Lunch1', 'scene1/scene1Bedroom2', 'scene2/scene2Bedroom', 'scene6/routeA/scene6EndingRouteA', 'deviceInfo', 'scene2/scene2Break', 'scene1/scene1Lunch2', 'scene4/scene4Backyard', 'computer/captions', 'menus/titleScene', 'computer/loginScreen', 'scene6/routeA/scene6LunchRouteA', 'computer/usernames

In [4]:
backend = Backend(name_mapping=lambda lng, ns: f"localization/{lng}/{ns}.json")

i18n = I18next(
	backend=backend,
	lng=list(settings.languages),
	ns=namespaces,
)



In [5]:
pattern = re.compile(r'<([^>]+)>')

def expand_variants(text: str):
	matches = pattern.findall(text)
	if not matches:
		return [text]
	
	sentences = [text]
	
	for match in matches:
		variants = [v.strip() for v in match.split(',')][1:]
		new_sentences = []
		for sentence in sentences:
			for var in variants:
				# Remplaza la primera ocurrencia
				new_sentence = pattern.sub(var, sentence, count=1)
				new_sentences.append(new_sentence)
		sentences = new_sentences
	
	return sentences

text = "Igualmente, <player, encantado, encantada> de <jugar, conocerte, conocer> *sonríes*"
expand_variants(text)


['Igualmente, encantado de conocerte *sonríes*',
 'Igualmente, encantado de conocer *sonríes*',
 'Igualmente, encantada de conocerte *sonríes*',
 'Igualmente, encantada de conocer *sonríes*']

In [6]:
def process_data(data):
	if isinstance(data, str):
		data = data.encode("latin1").decode("utf-8")
		return expand_variants(data)
	elif isinstance(data, list):
		results = []
		for obj in data:
			expanded = process_data(obj)
			expanded = expanded if isinstance(expanded, list) else [expanded]
			results.extend(expanded)
		return results
	elif isinstance(data, dict):
		return {k: process_data(v) for k, v in data.items()}
	else:
		return data
	
texts = i18n.t("part2.thanks2.responses", ns="scene1/scene1Classroom", return_objects=True)
fixed_texts = process_data(texts)
pprint(fixed_texts)


None


In [7]:
# rules = "one: n is 1; other:"
rules = {
	"one": "n is 1",
	"other": ""
}

plural_func = get_plural_func(rules)

print(plural_func(1))
print(plural_func(3))


one
other


In [8]:
base_dir = "./faiss_data"


In [9]:
lazy_sberts = get_sberts(settings.languages)
sberts = resolve_models(lazy_sberts)

def build_sbert_encoders(models: dict[str, Transformer]):
	return {
		lang: lambda sentences: model.encode(sentences, "mean")
		for lang, model in models.items()
	}

sbert_models = build_sbert_encoders(sberts)

sbert_engine = DenseVectorEngine(sbert_models, "sbert", base_dir)

engines = [
	sbert_engine
]


2026-04-20 05:47:17.939 | DEBUG    | controllers.load_models:model:23 - Loading SBERT for 'es'...


Using device: cuda


2026-04-20 05:47:25.825 | DEBUG    | controllers.load_models:model:29 - Successfully loaded SBERT for 'es'.


In [10]:
visited = set()

def build_full_id(language: str, filename: str, object_names: list[str], node_id: str):
	parts = [language, filename] + object_names + [node_id]
	return "_".join(parts)

def build_node_key(filename: str, object_names: list[str], node_id: str):
	parts = [filename] + object_names + [node_id]
	return "_".join(parts)

def build_localization_id(object_names: list[str], node_id: str):
	parts = object_names + [node_id]
	return ".".join(parts)

def extract_next_nodes(node: dict, loc_id: str, language: str, node_key: str):
	next_nodes = []
	node_type = node.get("type")

	if "next" in node:
		next_nodes.append(node["next"])
			
	elif node_type == "choice" and "choices" in node:
		for choice in node["choices"]:
			if "next" in choice:
				next_nodes.append(choice["next"])
	
	elif node_type == "similarity":
		if "choices" in node:
			key = f"{loc_id}.responses"
			responses = i18n.t(key, ns="scene1/scene1Classroom", return_objects=True, lng=language)
			fixed_responses = process_data(responses)			

			for engine in engines:
				engine.build_node(language, node_key, fixed_responses)
						
			for choice in node["choices"]:
				if "next" in choice:
					next_nodes.append(choice["next"])
					
		if "default" in node and "next" in node["default"]:
			next_nodes.append(node["default"]["next"])
	
	elif node_type == "condition" and "conditions" in node:
		for cond in node["conditions"]:
			if "next" in cond:
				next_nodes.append(cond["next"])
			
	return next_nodes

def dfs_traverse(language: str, filename: str, object_names: list[str], node_id: str, node_map: dict):
	full_id = build_full_id(language, filename, object_names, node_id)
	loc_id = build_localization_id(object_names, node_id)

	if full_id in visited:
		return

	visited.add(full_id)
	# print(full_id)

	node = node_map.get(node_id)
	if node:
		next_nodes = extract_next_nodes(node, loc_id, language, build_node_key(filename, object_names, node_id))
		for next_node in next_nodes:
			dfs_traverse(language, filename, object_names, next_node, node_map)

def traverse_graph(language: str, filename: str, object_names: list[str], node_map: dict):
	if "root" in node_map:
		dfs_traverse(language, filename, object_names, "root", node_map)
	else:
		for sub_name, sub_map in node_map.items():
			new_object_names = object_names + [sub_name]
			traverse_graph(language, filename, new_object_names, sub_map)
	

In [11]:
def run(base_path: str, languages: Iterable[str]):
	for root, _, files in os.walk(base_path):
		for file in files:
			if file.endswith(".json"):
				full_path = os.path.join(root, file)
				filename = os.path.splitext(os.path.basename(full_path))[0]

				with open(full_path, "r", encoding="utf-8") as f:
					data = json.load(f)

				for language in languages:
					if isinstance(data, dict) and "root" in data:
						traverse_graph(language, filename, [], data)

					elif isinstance(data, dict):
						for object_name, node_map in data.items():
							traverse_graph(language, filename, [object_name], node_map)

	print(f"Total visited nodes: {len(visited)}")

run("localization/structure", settings.languages)


for engine in engines:
	print(engine.retrievers)
	engine.save_all()


2026-04-20 05:47:26.643 | DEBUG    | controllers.faiss:fit:84 - Indexed 12 vectors
2026-04-20 05:47:26.668 | DEBUG    | services.dense_vector_engine:save_node:74 - Saving FAISS node | model=sbert | language=es | node=scene1Classroom_part2_thanks2


Total visited nodes: 670
{'es': {'scene1Classroom_part2_thanks2': <controllers.faiss.FaissRetriever object at 0x0000027BE414CC80>}}


In [12]:
test_engine = DenseVectorEngine(sbert_models, "sbert", base_dir)

print(test_engine.retrievers)

test_engine.load_all(settings.languages)

print(test_engine.retrievers)


2026-04-20 05:47:26.686 | DEBUG    | services.dense_vector_engine:load_node:88 - Loading FAISS node | model=sbert | language=es | node=scene1Classroom_part2_thanks2
2026-04-20 05:47:26.688 | SUCCESS  | services.dense_vector_engine:load_node:100 - Loaded node successfully.


{}
scene1Classroom_part2_thanks2
{'es': {'scene1Classroom_part2_thanks2': <controllers.faiss.FaissRetriever object at 0x0000027BE4ABA450>}}


In [13]:
retriever = test_engine.get_retriever("es", "scene1Classroom_part2_thanks2")

retriever.search("Si necesitas algo me dices", 3)


[SimilarityMatch(index=6, score=0.7370008230209351, text='Gracias, si necesito algo ya te iré diciendo.'),
 SimilarityMatch(index=7, score=0.6261634826660156, text='De acuerdo, ya te diré si necesito algo.'),
 SimilarityMatch(index=8, score=0.4190416634082794, text='Perfecto, muchas gracias por avisar.')]